In [20]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [25]:
import cv2
import numpy as np

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError("Não foi possível abrir a câmera.")

total = 0
objetos = {}
proximo_id = 0

while True:
    sucesso, frame = camera.read()
    if not sucesso:
        break

    frame = cv2.resize(frame, (640, 480))
    linha_y = 240

    # Detecta a cor preta em segundo plano
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mascara = cv2.inRange(
        hsv,
        np.array([0, 0, 0]),
        np.array([180, 255, 60])
    )

    # Remove pequenos ruídos
    kernel = np.ones((5, 5), np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel)

    contornos, _ = cv2.findContours(
        mascara, cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    deteccoes = []

    for contorno in contornos:
        if cv2.contourArea(contorno) < 1500:
            continue

        # Extrai as coordenadas do Bounding Box (x, y, largura, altura)
        x, y, w, h = cv2.boundingRect(contorno)
        deteccoes.append((x, y, w, h))

    # Associa cada produto ao objeto visto no quadro anterior
    usados = set()

    for x, y, w, h in deteccoes:
        cx = x + w // 2
        cy = y + h // 2

        melhor_id = None
        menor_distancia = 90

        for identificador, objeto in objetos.items():
            if identificador in usados:
                continue

            distancia = np.hypot(cx - objeto["cx"], cy - objeto["cy"])

            if distancia < menor_distancia:
                menor_distancia = distancia
                melhor_id = identificador

        if melhor_id is None:
            melhor_id = proximo_id
            proximo_id += 1

            objetos[melhor_id] = {
                "cx": cx,
                "cy": cy,
                "contado": False,
                "ausente": 0
            }

        objeto = objetos[melhor_id]

        # Conta quando o centro cruza a linha para baixo
        if not objeto["contado"] and objeto["cy"] < linha_y <= cy:
            total += 1
            objeto["contado"] = True

        objeto["cx"] = cx
        objeto["cy"] = cy
        objeto["ausente"] = 0
        usados.add(melhor_id)

        # --- DESENHO DO BOUNDING BOX DO PRODUTO ---
        # Muda a cor da caixa: Amarelo (não contado) -> Verde (já contado)
        cor_box = (0, 255, 0) if objeto["contado"] else (0, 255, 255)

        # 1. Caixa delimitadora (Bounding Box) ao redor do produto
        cv2.rectangle(frame, (x, y), (x + w, y + h), cor_box, 2)

        # 2. Ponto central do objeto
        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

        # 3. Etiqueta (Label) com fundo preenchido no topo do Bounding Box
        texto_label = f"Produto #{melhor_id} ({w}x{h})"
        (largura_texto, altura_texto), _ = cv2.getTextSize(
            texto_label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
        )

        # Garante que o texto não saia da tela se o objeto estiver muito no topo
        y_texto = max(y, altura_texto + 10)

        cv2.rectangle(
            frame,
            (x, y_texto - altura_texto - 10),
            (x + largura_texto + 6, y_texto),
            cor_box,
            -1  # Preenche o fundo da etiqueta
        )
        cv2.putText(
            frame, texto_label, (x + 3, y_texto - 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2
        )

    # Remove objetos que desapareceram
    for identificador in list(objetos):
        if identificador not in usados:
            objetos[identificador]["ausente"] += 1
            if objetos[identificador]["ausente"] > 10:
                del objetos[identificador]

    # Linha de contagem e placar total
    cv2.line(frame, (0, linha_y), (640, linha_y), (255, 0, 0), 2)
    cv2.putText(
        frame, f"TOTAL PRETOS: {total}", (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
    )

    # Exibe apenas a tela colorida com os Bounding Boxes
    cv2.imshow("Detector de Produtos - Q para sair", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()

print("Total de pretos:", total)

Total de pretos: 10
